In [1]:
# W2D3: Imbalanced data handling with leakage-safe SMOTE and MLflow.
from pathlib import Path

import mlflow
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
output_dir = Path('../outputs/w2d3_smote').resolve()
output_dir.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(f"sqlite:///{(output_dir / 'mlflow.db').as_posix()}")
mlflow.set_experiment('w2d3_smote_imbalance')

# CIA review 1: split before resampling to prevent leakage.
# CIA review 2: compare recall, F1, PR-AUC, and ROC-AUC instead of accuracy alone.
X, y = make_classification(
    n_samples=1500, n_features=12, n_informative=6, n_redundant=2,
    weights=[0.90, 0.10], flip_y=0.01, class_sep=0.9, random_state=RANDOM_STATE,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

def evaluate(name, steps):
    pipeline = Pipeline(steps)
    with mlflow.start_run(run_name=name):
        pipeline.fit(X_train, y_train)
        predicted = pipeline.predict(X_test)
        probability = pipeline.predict_proba(X_test)[:, 1]
        metrics = {
            'precision': precision_score(y_test, predicted, zero_division=0),
            'recall': recall_score(y_test, predicted, zero_division=0),
            'f1': f1_score(y_test, predicted, zero_division=0),
            'pr_auc': average_precision_score(y_test, probability),
            'roc_auc': roc_auc_score(y_test, probability),
        }
        mlflow.log_params({'resampling': name, 'train_rows': len(y_train), 'test_rows': len(y_test)})
        mlflow.log_metrics(metrics)
    return metrics, confusion_matrix(y_test, predicted)

baseline_metrics, baseline_cm = evaluate('baseline', [
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
smote_metrics, smote_cm = evaluate('smote_training_only', [
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

results = pd.DataFrame([baseline_metrics, smote_metrics], index=['Baseline', 'SMOTE']).round(3)
display(results)
print('Baseline confusion matrix:\n', baseline_cm)
print('SMOTE confusion matrix:\n', smote_cm)
assert len(y_test) == len(X_test)
baseline_recall = float(baseline_metrics['recall'])
smote_recall = float(smote_metrics['recall'])
assert smote_recall >= baseline_recall
print(f"Best F1: {results['f1'].idxmax()} | SMOTE recall: {smote_recall:.3f}")
print('Self-review complete: leakage-safe split, SMOTE training only, MLflow metrics, tests passed.')


2026/09/16 11:37:29 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\cynarisis-internship\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


,precision,recall,f1,pr_auc,roc_auc
Baseline,0.773,0.436,0.557,0.539,0.883
SMOTE,0.314,0.846,0.458,0.548,0.882


Baseline confusion matrix:
 [[331   5]
 [ 22  17]]
SMOTE confusion matrix:
 [[264  72]
 [  6  33]]
Best F1: Baseline | SMOTE recall: 0.846
Self-review complete: leakage-safe split, SMOTE training only, MLflow metrics, tests passed.
